# Controlled-pilot engineering vertical slice

This notebook demonstrates the new compressor case envelope and fail-closed production-slice qualification. It intentionally uses an incomplete safety topology so the final result shows actionable failed gates. Replace the synthetic inputs with controlled project cases, HAZOP/LOPA/SRS evidence, vendor maps and approved rules. A passing result qualifies only a controlled pilot; it is never construction approval.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path(os.environ.get('NEQSIM_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'pom.xml').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'devtools'))
from neqsim_dev_setup import neqsim_init, neqsim_classes
ns = neqsim_classes(neqsim_init(project_root=ROOT, recompile=not (ROOT / 'target/classes').exists(), verbose=False))
import jpype
JClass = ns.JClass
print('Loaded NeqSim from', ROOT)


## 1. Create the simulated inlet/compression/export core

The compressor uses an explicit performance map plus surge and stonewall curves. The qualification policy later requires the cooler, recycle, valves and flare topology as well; omitting them here demonstrates that missing engineering cannot pass silently.


In [ ]:
System = JClass('neqsim.thermo.system.SystemSrkEos')
Stream = JClass('neqsim.process.equipment.stream.Stream')
Separator = JClass('neqsim.process.equipment.separator.Separator')
Compressor = JClass('neqsim.process.equipment.compressor.Compressor')
Pipe = JClass('neqsim.process.equipment.pipeline.AdiabaticPipe')
ProcessSystem = JClass('neqsim.process.processmodel.ProcessSystem')
SurgeCurve = JClass('neqsim.process.equipment.compressor.SafeSplineSurgeCurve')
StonewallCurve = JClass('neqsim.process.equipment.compressor.SafeSplineStoneWallCurve')

fluid = System(300.0, 50.0)
fluid.addComponent('methane', 0.92); fluid.addComponent('ethane', 0.04); fluid.addComponent('n-heptane', 0.04)
fluid.setMixingRule('classic')
feed = Stream('FEED', fluid); feed.setFlowRate(8000.0, 'kg/hr')
separator = Separator('INLET-SEP', feed)
compressor = Compressor('EXPORT-COMP', separator.getGasOutStream())
compressor.setOutletPressure(80.0, 'bara'); compressor.setSpeed(10000.0)
line = Pipe('EXPORT-LINE', compressor.getOutletStream()); line.setLength(1000.0); line.setDiameter(0.2027)
process = ProcessSystem()
for unit in [feed, separator, compressor, line]: process.add(unit)
process.run()

JDoubleArray = jpype.JArray(jpype.JDouble)
JDoubleMatrix = jpype.JArray(JDoubleArray)
compressor.getCompressorChart().setCurves(
    JDoubleArray([19.0, 300.0, 50.0, 0.90]), JDoubleArray([9000.0, 11000.0]),
    JDoubleMatrix([JDoubleArray([50.0, 100.0, 150.0]), JDoubleArray([70.0, 140.0, 210.0])]),
    JDoubleMatrix([JDoubleArray([75.0, 65.0, 50.0]), JDoubleArray([110.0, 95.0, 72.0])]),
    JDoubleMatrix([JDoubleArray([70.0, 79.0, 73.0]), JDoubleArray([71.0, 80.0, 74.0])]))
compressor.getCompressorChart().setSurgeCurve(SurgeCurve(JDoubleArray([42.0, 55.0, 70.0]), JDoubleArray([110.0, 90.0, 65.0])))
compressor.getCompressorChart().setStoneWallCurve(StonewallCurve(JDoubleArray([170.0, 195.0, 220.0]), JDoubleArray([110.0, 90.0, 65.0])))
compressor.getCompressorChart().setHeadUnit('kJ/kg'); compressor.getCompressorChart().setUseCompressorChart(True)
compressor.getAntiSurge().setActive(True)


## 2. Declare controlled cases and policies

`VerticalSliceCaseMatrixFactory` provides a JPype-friendly builder for explicit scalar boundary conditions. It does not invent transient event actions or scenario credibility: those remain separate dynamic-scenario and HAZOP inputs.


In [ ]:
Builder = JClass('neqsim.process.engineering.NorsokOffshoreEngineeringBuilder')
AutoPolicy = JClass('neqsim.process.engineering.production.EngineeringAutoConfigurationPolicy')
QualificationPolicy = JClass('neqsim.process.engineering.verticalslice.InletCompressionExportSlicePolicy')
CaseMatrix = JClass('neqsim.process.engineering.verticalslice.VerticalSliceCaseMatrixFactory')
CaseType = JClass('neqsim.process.engineering.designcase.EngineeringDesignCase$Type')
Evidence = JClass('neqsim.process.engineering.EngineeringEvidenceRecord')
project = Builder.from_('Controlled pilot example', process).projectId('compression-pilot').build()

design_policy = (AutoPolicy('compression-design', 'A')
    .addInletCompressionExportSlice('INLET-SEP', 'EXPORT-COMP', 'EXPORT-LINE', '', 'PIT-100',
        800.0, 0.107, 120.0, 25.0, 5.0, 0.10, JDoubleArray([500.0, 1000.0, 2000.0, 5000.0, 10000.0]))
    .addCompressorOperatingEnvelope('EXPORT-COMP', 0.05, 0.05, 160.0, 0.10))
qualification_policy = (QualificationPolicy.builder('compression-pilot', 'A')
    .processTags('INLET-SEP', 'EXPORT-COMP', 'AFTERCOOLER', 'EXPORT-LINE')
    .controlTags('ASV', 'ASV-RECYCLE', 'PCV', 'LCV')
    .safetyTags('PSV', 'BDV', 'ESDV-SUCTION', 'ESDV-DISCHARGE', 'FLARE-CONNECTION')
    .addRequiredDynamicScenario('compressor-trip-esd')
    .addEvidenceReference('SYNTHETIC-HAZOP-REQUIRED').build())
project.addEvidenceRecord(Evidence('SYNTHETIC-HAZOP-REQUIRED', 'HAZOP', 'A')
    .setTitle('Synthetic notebook hazard review placeholder')
    .setSourceOrganization('Notebook example').linkRequirement('VERTICAL-SLICE'))
case_matrix = CaseMatrix.builder('FEED', 'EXPORT-COMP')
case_specs = [('normal', CaseType.NORMAL, 8000.0), ('turndown', CaseType.MINIMUM_TURNDOWN, 5000.0),
    ('maximum', CaseType.MAXIMUM_PRODUCTION, 10000.0), ('startup', CaseType.STARTUP, 6000.0),
    ('shutdown', CaseType.SHUTDOWN, 4000.0), ('trip', CaseType.EQUIPMENT_TRIP, 4000.0),
    ('settle-out', CaseType.SETTLE_OUT, 1000.0), ('blocked-outlet', CaseType.BLOCKED_OUTLET, 1000.0),
    ('fire', CaseType.FIRE, 1000.0), ('blowdown', CaseType.BLOWDOWN, 1000.0)]
for case_id, case_type, flow in case_specs:
    case_matrix.addCase(case_id, case_id, case_type, flow, 50.0, 26.85, 80.0,
        'SYNTHETIC-HAZOP-REQUIRED', 'REVIEW_REQUIRED')
case_matrix.applyTo(project, qualification_policy)
print('Policies created:', design_policy.getId(), qualification_policy.getPolicyId())


## 3. Preflight before numerical execution

Preflight is intentionally safe to run on this incomplete example: it performs no thermodynamic or dynamic calculation. The blockers are the exact controlled inputs that must be completed before strict execution.


In [ ]:
Preflight = JClass('neqsim.process.engineering.verticalslice.ProductionVerticalSlicePreflight')
preflight = Preflight.assess(project, qualification_policy)
print('Ready for simulation:', preflight.isReadyForSimulation())
print('Failed checks:')
for check in preflight.getFailedChecks(): print(' -', check)
print('First blockers:')
for blocker in list(preflight.getBlockers())[:10]: print(' -', blocker)


## 4. Strict run, qualification and package compilation

After closing every preflight blocker, call `runStrictAndCompile(...)`. Strict mode refuses to start the engineering loop if the case matrix, topology, maps, scenarios, flare studies, standards or evidence are incomplete. Inspect the qualification and retain `engineering-vertical-slice-execution-manifest.json`; its SHA-256 fingerprint changes whenever controlled execution inputs change.

```java
ProductionVerticalSliceSimulator.Result result = ProductionVerticalSliceSimulator.runStrictAndCompile(
    project, designPolicy, qualificationPolicy, 4, outputDirectory, baselineGraph);
System.out.println(result.getManifest().getInputFingerprint());
System.out.println(result.getQualification().getFailedGates());
```


## Required interpretation

A fully passing artifact sets only `qualifiedForControlledPilot=true`. `qualifiedForFeedSupport`, `fitnessForConstruction`, and `finalEngineeringApprovalGranted` remain false. Independent benchmarks, commercial DEXPI round trips, accepted pilots, safety-lifecycle approval and accountable engineering approval remain separate gates.
